# Notebook 4 — Build a Supply Chain Agent with Databricks Tools

This extends the simple laptop/server hybrid RAG example into an **agent-tool workflow**, following the uploaded Databricks Academy lab **“3.5L — Creating Tools and Testing in the Playground.”**

Tools:
1. `search_item_master` — semantic search through the Item Master Vector Search index.
2. `get_inventory` — exact stock by item and warehouse.
3. `get_supplier` — supplier name, supplier location, and delivery lead time.

This notebook creates Unity Catalog functions to add as hosted-function tools in Databricks AI Playground. The model decides when to call them in Playground; this notebook does not itself deploy an agent endpoint.

Run Notebook 1 first and Notebook 3 to create the Vector Search index and bring it online. Use fictional training data only.


## 1. Configure catalog, schema, and index

Use the same catalog/schema as your previous notebooks. The Vector Search index must exist and be online before testing semantic search.

In [0]:
catalog_name = "main"  # Change to a catalog you can use
schema_name = "scm_e2e_rag_lab"

item_table = f"{catalog_name}.{schema_name}.item_master"
inventory_table = f"{catalog_name}.{schema_name}.inventory"
supplier_table = f"{catalog_name}.{schema_name}.supplier"
index_name = f"{catalog_name}.{schema_name}.item_master_vector_index"

print("Catalog:", catalog_name)
print("Schema:", schema_name)
print("Vector index:", index_name)


## 2. Create a structured tool: `get_inventory`

Answers exact questions such as “How many SSD001 items are available in Chennai?”

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.get_inventory(
  p_item_id STRING COMMENT 'Item ID, for example SSD001',
  p_warehouse STRING COMMENT 'Warehouse city, for example Chennai'
)
RETURNS TABLE (
  item_id STRING,
  warehouse STRING,
  available_quantity BIGINT
)
COMMENT 'Returns exact available inventory for an item at a specified warehouse'
LANGUAGE SQL
RETURN
  SELECT
    i.item_id,
    i.warehouse,
    CAST(i.available_quantity AS BIGINT) AS available_quantity
  FROM {inventory_table} AS i
  WHERE i.item_id = p_item_id
    AND i.warehouse = p_warehouse
""")


### Test the inventory tool

Expected example: `SSD001` in `Chennai` has 50 units in the sample dataset.

In [0]:
display(spark.sql(
    f"SELECT * FROM {catalog_name}.{schema_name}.get_inventory('SSD001', 'Chennai')"
))


## 3. Create a structured tool: `get_supplier`

Answers who supplies an item, where the supplier is located, and the delivery lead time. Supplier location is different from warehouse location.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.get_supplier(
  p_item_id STRING COMMENT 'Item ID, for example SSD001'
)
RETURNS TABLE (
  item_id STRING,
  supplier_name STRING,
  supplier_city STRING,
  supplier_country STRING,
  lead_time_days INT
)
COMMENT 'Returns supplier name, location, and delivery lead time for an item'
LANGUAGE SQL
RETURN
  SELECT
    s.item_id,
    s.supplier_name,
    s.supplier_city,
    s.supplier_country,
    CAST(s.lead_time_days AS INT) AS lead_time_days
  FROM {supplier_table} AS s
  WHERE s.item_id = p_item_id
""")


### Test the supplier tool

Expected example: SSD001 is supplied by ABC Components in Chennai, India, with a 7-day lead time.

In [0]:
display(spark.sql(
    f"SELECT * FROM {catalog_name}.{schema_name}.get_supplier('SSD001')"
))


## 4. Create an unstructured tool: `search_item_master`

This uses SQL `vector_search(...)` and `ai_query(...)`, following the pattern in the supplied Databricks Academy lab.

Prerequisites:
- The index `main.scm_rag_lab.item_master_vector_index` (or your configured equivalent) exists and is online.
- The embedding endpoint `databricks-gte-large-en` is available.
- Your workspace supports SQL `vector_search` and `ai_query`.


In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog_name}.{schema_name}.search_item_master(
  search_query STRING COMMENT 'Natural-language description of an item to find'
)
RETURNS TABLE (
  item_id STRING,
  item_document STRING
)
COMMENT 'Finds semantically similar item master records using Databricks Vector Search'
RETURN (
  SELECT
    id AS item_id,
    document AS item_document
  FROM vector_search(
    index => '{index_name}',
    query_vector => ai_query('databricks-qwen3-embedding-0-6b', search_query, 'STRING'),
    num_results => 3
  )
)
""")


### Test the semantic search tool

Expected to retrieve relevant records for a phrase like “solid state drive for a laptop.” Exact results depend on the index and embedding model.

In [0]:
display(spark.sql(
    f"SELECT * FROM {catalog_name}.{schema_name}.search_item_master('solid state drive for a laptop.')"
))


In [0]:
spark.sql(f"select * from main.scm_e2e_rag_lab.item_master").show(truncate=False)
spark.sql(f"select * from main.scm_e2e_rag_lab.inventory").show(truncate=False)
spark.sql(f"select * from main.scm_e2e_rag_lab.supplier").show(truncate=False)

## 5. Add the functions as tools in Databricks AI Playground

This follows the uploaded Academy lab. The functions are tools; the model performs tool calling in Playground.

1. Open **AI Playground** in your Databricks workspace.
2. Select a model that supports tool calling.
3. Open **Tools** → **Add** → **Add tool**.
4. Choose **UC Function** → **Add hosted function**.
5. Add these three functions:
   - `main.scm_rag_lab.search_item_master`
   - `main.scm_rag_lab.get_inventory`
   - `main.scm_rag_lab.get_supplier`
6. Save the tools and test the prompts below.

Use your actual catalog/schema if different. If a function does not appear, verify it was created successfully and that your user has permission to execute it. Workspace UI labels can vary by release.


## 6. Test prompts for the agent

**Semantic search**
- “Find an item similar to a solid state drive for a laptop.”
- “Find server memory.”

**Structured inventory**
- “How many SSD001 units are available in Chennai?”
- “Check RAM002 stock in Chennai.”

**Supplier details**
- “Who supplies SSD001, where is the supplier located, and how long is delivery?”

**Multi-tool question**
- “Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time.”

For the last question, the agent may call `search_item_master`, then use the returned item ID with `get_inventory` and `get_supplier`. Inspect the tool-call trace; the exact sequence depends on the model.


**Agent Test Flow**
![image_1792145489027.png](./image_1792145489027.png "image_1792145489027.png")
![image_1792145496256.png](./image_1792145496256.png "image_1792145496256.png")

## 7. Suggested agent instructions for Playground

> You are a supply-chain assistant for a fictional laptop and server manufacturer. Use `search_item_master` to find relevant item records by meaning. Use `get_inventory` for exact quantities at a named warehouse. Use `get_supplier` for supplier name, supplier city, supplier country, and lead time. Do not invent missing facts. If a required item ID or warehouse is missing, ask a clarifying question. Supplier location and warehouse location are different. Similarity does not prove technical compatibility; do not claim an item is a compatible substitute unless the data explicitly establishes that.



## 8. Troubleshooting and checkpoint

- **Function creation fails:** Check permissions to create Unity Catalog functions in the schema.
- **Inventory/supplier returns no rows:** Check item ID and warehouse spelling against Notebook 1.
- **Semantic search fails:** Confirm the index is online and SQL `vector_search`, `ai_query`, and the embedding endpoint are available.
- **Agent does not call a tool:** Confirm the tool is attached in Playground and the selected model supports tool calling.
- **Permission errors:** Apply your workspace's minimum required catalog, schema, function, and table permissions.

### What changed from RAG to Agent?

- In the earlier RAG notebook, Python explicitly called Vector Search and database functions.
- Here, those operations are exposed as tools.
- The LLM can choose which tools to call based on the user's question and use the returned results to answer.

This notebook creates and tests the tools. To export a standalone agent program or deploy it, follow the supplied Academy lab's **Get code → Create Agent Notebook** workflow, then adapt the generated code to these functions.
